# Phase 1: Data Exploration & Initial Dataset Analysis

**Project**: Personalized Movie Recommendation System  
**Repository**: `Dp8453/Personalized-movie-recommendation-system`  
**Dataset**: TMDB 5000 Movie & Credits Dataset  

---  
### Objectives of Phase 1 EDA:
1. Load and inspect merged raw dataset (`movies.csv` + `credits.csv`) using primary key merging (`id` == `movie_id`).
2. Examine dataset dimensions, columns, data types, missing values, and duplicates.
3. Perform statistical analysis on numerical fields (`vote_average`, `vote_count`, `runtime`, `budget`, `revenue`).
4. Evaluate metadata fields (`genres`, `keywords`, `cast`, `crew`, `overview`, `original_language`) for recommendation potential.
5. Document findings and identify Phase 2 preprocessing requirements.

## 1. Import Required Libraries

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure src module can be imported from parent directory
sys.path.append(os.path.abspath('..'))
from src.data_loader import load_movies

# Set display options for thorough inspection
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
print('Libraries imported successfully.')

## 2. Load Dataset using `src.data_loader.load_movies()`

In [ ]:
df = load_movies()
print(f'Merged Dataset Shape: {df.shape}')

### 2.1 First 5 Rows Preview

In [ ]:
df.head()

### 2.2 Last 5 Rows Preview

In [ ]:
df.tail()

## 3. Dataset Shape, Columns, and Data Types

In [ ]:
print(f'Total Rows (Movies): {df.shape[0]}')
print(f'Total Columns: {df.shape[1]}\n')
print('Column Names and Data Types:')
print(df.dtypes)

## 4. Primary Key Alignment & Duplicate Inspection

In [ ]:
# Verify primary key alignment
id_matches = (df['id'] == df['movie_id']).sum()
print(f"Primary Key Match Count ('id' == 'movie_id'): {id_matches} / {len(df)}")

# Missing value count and percentage
missing_df = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage (%)': (df.isnull().sum() / len(df)) * 100
})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values(by='Missing_Count', ascending=False)
print('\nColumns with Missing Values:')
print(missing_df)

### 4.1 Check Duplicate Movies

In [ ]:
duplicate_rows = df.duplicated().sum()
duplicate_titles = df.duplicated(subset=['title']).sum()
duplicate_movie_ids = df.duplicated(subset=['movie_id']).sum()

print(f'Duplicate Entire Rows: {duplicate_rows}')
print(f'Duplicate Movie IDs: {duplicate_movie_ids}')
print(f'Duplicate Movie Titles: {duplicate_titles}')

if duplicate_titles > 0:
    print('\nSample Duplicate Movie Titles (Remakes/Re-releases):')
    print(df[df.duplicated(subset=['title'], keep=False)][['id', 'title', 'release_date']])

## 5. Descriptive Statistics

In [ ]:
df.describe().T

## 6. Detailed Feature Investigation for Recommendation

In [ ]:
important_cols = ['id', 'title', 'genres', 'keywords', 'cast', 'crew', 'overview', 'vote_average', 'vote_count', 'original_language']
print('Unique counts for key columns:')
for col in important_cols:
    print(f'  {col:20s}: {df[col].nunique()} unique values')

### 6.1 Sample Metadata Contents (`genres`, `keywords`, `cast`, `crew`, `overview`)

In [ ]:
sample_movie = df.iloc[0]
print(f"ID: {sample_movie['id']} | Title: {sample_movie['title']}\n")
print(f"Overview:\n{sample_movie['overview']}\n")
print(f"Raw Genres JSON:\n{sample_movie['genres']}\n")
print(f"Raw Keywords JSON:\n{sample_movie['keywords']}\n")
print(f"Raw Cast JSON (Truncated):\n{str(sample_movie['cast'])[:200]}...\n")
print(f"Raw Crew JSON (Truncated):\n{str(sample_movie['crew'])[:200]}...")

## 7. Comprehensive Dataset Analysis & Phase 1 Conclusions

Based on the empirical calculations above, here are the answers to the key dataset analysis questions:

### Q1. How many movies are available?
- **4,803 unique movie records** (resulting from inner-joining `tmdb_5000_movies.csv` and `tmdb_5000_credits.csv` on primary keys `id` == `movie_id`).
- *Note on Merge Key*: Merging on `title` produces 4,809 records due to duplicate title collisions (e.g. *Batman* 1966 vs 1989). Merging on primary key `id` == `movie_id` maintains exact 1-to-1 matching across all 4,803 movies.

### Q2. What are the important columns?
- `id` & `movie_id`: Stable primary key integer identifiers.
- `title`: Official movie title.
- `overview`: Plot synopsis (core narrative text representation).
- `genres`: High-level thematic categories.
- `keywords`: Detailed plot keywords.
- `cast`: Actors and lead performers.
- `crew`: Production crew (specifically `Director`).
- `vote_average` & `vote_count`: Quality and popularity indicators.

### Q3. Which columns contain missing values?
- `homepage`: 3,091 missing values (64.36% null) — *Irrelevant for recommendation*.
- `tagline`: 844 missing values (17.57% null) — *Low quality / sparse text*.
- `overview`: 3 missing values (0.06% null) — *Will be imputed with empty string in Phase 2*.
- `runtime`: 2 missing values (0.04% null).
- `release_date`: 1 missing value (0.02% null).

### Q4. Are there duplicate movies?
- **0 duplicate entire rows**.
- **0 duplicate `movie_id` or `id` values** (each movie entry has a unique integer primary key).
- **3 duplicate title pairs (6 entries)** for remakes/re-releases (*Batman*, *The Host*, *Out of the Blue*).

### Q5. Which metadata fields are useful for content-based recommendation?
1. `overview` — Narrative plot summary.
2. `genres` — High-level movie category.
3. `keywords` — Detailed thematic keywords.
4. `cast` — Top 3 lead actors.
5. `crew` — Director name.

### Q6. Which fields are unsuitable or unnecessary?
- `budget`, `revenue`, `homepage`, `production_companies`, `production_countries`, `spoken_languages`, `status`, `tagline`, `popularity` — These do not directly capture content similarity or contain high noise/missingness.

### Q7. What preprocessing problems will need to be solved in Phase 2?
1. **JSON String Parsing**: Extract string values from stringified JSON lists (`ast.literal_eval`).
2. **Cast Extraction**: Filter top 3 lead actors from full cast list.
3. **Crew Filtering**: Extract director name where `job == 'Director'`.
4. **Space Removal**: Convert multi-word strings to single tokens (e.g., `"Johnny Depp"` -> `"JohnnyDepp"`) so TF-IDF treats full names as unified concepts.
5. **Handling Nulls**: Replace missing `overview` entries with empty strings.
6. **Text Concatenation**: Combine cleaned `overview`, `genres`, `keywords`, `cast`, and `crew` into a single `tags` feature column.